# GLOSA Traffic Engine: PeMS Data Extraction Pipeline

**Description:** This notebook processes raw 5-minute vehicular telemetry data from the Caltrans Performance Measurement System (PeMS). It iterates through 1 week's worth of data, filtering strictly for I-280 and SR-85 off-ramps to build a localized traffic demand dataset for Cupertino, California.

### PeMS Data Dictionary (Extracted Columns)

This dataset isolates the 6 core metrics required for the traffic simulation:

* **0 - Timestamp:** Date and time of the sensor reading (5-minute intervals).
* **1 - Station ID:** Unique identifier for the physical off-ramp sensor.
* **3 - Freeway:** Highway number (280 or 85).
* **4 - Direction:** Compass direction of travel (N, S, E, W).
* **5 - Lane Type:** Classification of the lane (`FR` = Off-Ramp).
* **9 - Flow:** Total vehicle volume passing the sensor in the 5-minute window.

In [1]:
# Step 1: Workspace setup

import numpy as np
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt

!pip install plotly express

ERROR: Could not find a version that satisfies the requirement express (from versions: none)
ERROR: No matching distribution found for express


In [2]:
# Step 2: Create master list for all weeks data & final master_data file

traffic_data_week = []
master_data = []

# Step 3: Define constants
# I am only going to work on freeways that are feeding into and out of cupertino, which are 280 and 85.
# The target lane type will be 'FR' which is 'Off-Ramp' to understand traffic patters on the streets.

TARGET_FREEWAYS = [280, 85]
TARGET_LANE_TYPE = ['FR']

In [3]:
# Step 4: Exploratory Data Analysis (EDA)
"""
Before building the full extraction loop, we need to inspect a single compressed file. 
Since the raw data lacks column headers, we will load one day into a Pandas DataFrame to determine the exact string formatting Caltrans uses for highway numbers and lane types.
"""

df_test = pd.read_csv('Data/jul10.txt.gz', header=None)

In [4]:
df_test.sample(5)

,0,1,2,3,4,5,6,7,8,9,...,42,43,44,45,46,47,48,49,50,51
2171,07/10/2026 00:00:00,406553,4,101,N,ML,0.485,30,100,63.0,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0
931970,07/10/2026 19:50:00,408865,4,101,S,OR,NaN,20,100,12.0,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0
924378,07/10/2026 19:40:00,410156,4,101,S,OR,NaN,10,100,9.0,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0
332897,07/10/2026 07:05:00,401935,4,101,S,ML,1.215,40,0,424.0,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0
96652,07/10/2026 02:00:00,411149,4,80,W,ML,0.200,40,75,41.0,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0


### Validating Target Variables
By extracting the unique values from Column 3 (Freeway) and Column 5 (Lane Type), we can verify the exact syntax required for our filtering logic. The describe() method provides a quick statistical overview to ensure the numerical data (like flow and speed) loaded correctly.

In [5]:
df_test[3].unique()

array([101, 880,  80,  85, 680, 580, 280,  87,  24,   4, 238,  92,  17,
       237, 242, 948, 980, 780,  25,  37,  29, 160, 156,  12, 152,   1,
        84,  13, 380, 221])

In [6]:
df_test[5].unique()

array(['ML', 'FF', 'FR', 'OR', 'HV'], dtype=object)

In [7]:
df_test.describe()

,1,2,3,6,7,8,9,10,11,12,...,42,43,44,45,46,47,48,49,50,51
count,1.124640e+06,1124640.0,1.124640e+06,660960.000000,1.124640e+06,1.124640e+06,920803.000000,920803.000000,660384.000000,1.124640e+06,...,5472.000000,5472.000000,5472.000000,5472.000000,1.124640e+06,288.0,288.0,288.0,288.000000,1124640.0
mean,4.083034e+05,4.0,2.916374e+02,0.610276,2.580912e+01,5.287325e+01,190.750720,0.052809,65.270024,8.962531e+00,...,9.467836,39.457785,0.046588,62.613469,1.792574e-03,0.0,0.0,0.0,58.998958,0.0
std,8.695981e+03,0.0,3.048638e+02,0.564434,1.621128e+01,4.742433e+01,174.434664,0.055256,6.825272,3.005533e+00,...,2.233424,26.377524,0.050280,4.688666,4.230085e-02,0.0,0.0,0.0,6.276262,0.0
min,4.000000e+05,4.0,1.000000e+00,0.015000,0.000000e+00,0.000000e+00,0.000000,0.000000,3.000000,0.000000e+00,...,0.000000,0.000000,0.000000,5.300000,0.000000e+00,0.0,0.0,0.0,35.700000,0.0
25%,4.019430e+05,4.0,8.000000e+01,0.315000,1.000000e+01,0.000000e+00,36.000000,0.016700,63.300000,1.000000e+01,...,10.000000,22.000000,0.016400,62.000000,0.000000e+00,0.0,0.0,0.0,56.800000,0.0
50%,4.045860e+05,4.0,1.010000e+02,0.450000,2.000000e+01,6.700000e+01,136.000000,0.042500,66.400000,1.000000e+01,...,10.000000,37.000000,0.035250,63.300000,0.000000e+00,0.0,0.0,0.0,59.550000,0.0
75%,4.111260e+05,4.0,5.800000e+02,0.665000,4.000000e+01,1.000000e+02,323.000000,0.072400,68.700000,1.000000e+01,...,10.000000,53.000000,0.057500,64.500000,0.000000e+00,0.0,0.0,0.0,64.100000,0.0
max,4.329560e+05,4.0,9.800000e+02,5.000000,7.000000e+01,1.000000e+02,1077.000000,0.998900,85.400000,1.000000e+01,...,10.000000,224.000000,0.648300,74.500000,1.000000e+00,0.0,0.0,0.0,65.300000,0.0


In [8]:
df_test.isnull().sum()

0           0
1           0
2           0
3           0
4           0
5           0
6      463680
7           0
8           0
9      203837
10     203837
11     464256
12          0
13     216257
14     216257
15     464256
16          0
17     262944
18     337874
19     337874
20     471456
21          0
22     522720
23     535105
24     535105
25     559584
26          0
27     705024
28     705024
29     705024
30     705312
31          0
32     963648
33     963648
34     963648
35     963648
36          0
37    1084320
38    1084320
39    1084320
40    1084320
41          0
42    1119168
43    1119168
44    1119168
45    1119168
46          0
47    1124352
48    1124352
49    1124352
50    1124352
51          0
dtype: int64

In [9]:
# Look at the first 12 columns for a random sample of 5 rows
df_test.iloc[:, 0:12].head(5)

,0,1,2,3,4,5,6,7,8,9,10,11
0,07/10/2026 00:00:00,400000,4,101,S,ML,0.415,40,100,38.0,0.0070,69.0
1,07/10/2026 00:00:00,400001,4,101,N,ML,0.265,50,0,72.0,0.0137,66.1
2,07/10/2026 00:00:00,400002,4,101,S,ML,0.305,50,0,217.0,0.0442,71.0
3,07/10/2026 00:00:00,400006,4,880,S,ML,0.340,40,75,125.0,0.0255,67.0
4,07/10/2026 00:00:00,400007,4,101,N,ML,0.365,50,0,117.0,0.0203,69.9


# Step 5: Data Extraction Pipeline

We iterate through the 7 `.gz` files to build our localized weekly dataset. To optimize memory, we extract only the 6 essential columns (Timestamp, Station ID, Freeway, Direction, Lane Type, Flow). 

Using Pandas boolean masking, we filter strictly for our Cupertino targets (Freeways 280 & 85, Off-Ramps), appending each cleaned day into a single, lightweight master DataFrame.

In [10]:
# Extract data for an entire week with only information we need

traffic_data_week = []

for i in range(10, 17):
    df_week = pd.read_csv(f'Data/jul{i}.txt.gz', header=None)
    df_week = df_week[[0, 1, 3, 4, 5, 9]]
    df_week = df_week[(df_week[3].isin(TARGET_FREEWAYS)) & (df_week[5].isin(TARGET_LANE_TYPE))]

    traffic_data_week.append(df_week)

master_data = pd.concat(traffic_data_week, ignore_index=True)

In [11]:
master_data.dropna(subset=[9], inplace=True)

In [12]:
# Rename the columns for readability
master_data.columns = ['Timestamp', 'Station_ID', 'Freeway', 'Direction', 'Lane_Type', 'Flow']

master_data.sample(10)

,Timestamp,Station_ID,Freeway,Direction,Lane_Type,Flow
35222,07/14/2026 01:50:00,403415,280,S,FR,4.0
53646,07/16/2026 05:00:00,407327,85,N,FR,13.0
40946,07/14/2026 17:40:00,425245,280,N,FR,63.0
35118,07/14/2026 01:30:00,410590,280,N,FR,11.0
6103,07/10/2026 16:55:00,407355,85,S,FR,25.0
16986,07/11/2026 23:10:00,407327,85,N,FR,42.0
8538,07/10/2026 23:40:00,410590,280,N,FR,24.0
116,07/10/2026 00:15:00,425245,280,N,FR,6.0
17096,07/11/2026 23:25:00,425245,280,N,FR,9.0
35703,07/14/2026 03:10:00,403416,280,S,FR,0.0


In [16]:
master_data.shape

(14400, 6)

In [14]:
import plotly.express as px

# Create an interactive line chart
fig = px.line(
    master_data, 
    x='Timestamp', 
    y='Flow', 
    color='Freeway',
    title='Cupertino Off-Ramp Traffic Flow (July 10-16)',
    labels={'Flow': 'Vehicles per 5 Mins', 'Timestamp': 'Date & Time'}
)

# Show the interactive plot
fig.show()

In [15]:
# Create a copy of the essential columns
sumo_data = master_data[['Station_ID', 'Timestamp', 'Flow']].copy()

# Rename columns to match dfrouter's exact expected syntax
sumo_data.columns = ['Detector', 'Time', 'Flow']

# Add a dummy Speed column (required by SUMO, but 0 is fine for off-ramps)
sumo_data['Speed'] = 0

# Export to a semicolon-separated CSV file
sumo_data.to_csv('cupertino_traffic_flow.csv', sep=';', index=False)

print("SUMO traffic data successfully generated!")

SUMO traffic data successfully generated!
